In [ ]:
# Cell 1: Install vLLM (Replaces Ollama)
!pip install vllm==0.6.4.post1 transformers==4.45.2 pandas tqdm


In [ ]:
import os
import sys
import transformers

# Verify we are NOT on v5.x
print(f"Transformers version: {transformers.__version__}")

# Existing fixes
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1
if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2

print("✅ System fixes applied.")

In [ ]:
# Cell 2: Setup and Drive Mount
from google.colab import drive
import pandas as pd
import json
import os
import time
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

drive.mount('/content/drive')

# --- CONFIGURATION ---
DATASET_PATH = "/content/drive/MyDrive/SFT/Datasets/beavertails_unsafe.csv"
PROMPT_COLUMN = "prompt"
OUTPUT_FILE = "/content/drive/MyDrive/SFT/Processed datasets/beavertails_unsafe_vllm.json"
MODEL_NAME = "TheBloke/Mistral-7B-Instruct-v0.2-AWQ"

# We must use enforce_eager and limit max_model_len to fit the T4 GPU (16GB VRAM)
llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    dtype="half",
    enforce_eager=True,
    gpu_memory_utilization=0.9, # Increased slightly
    max_model_len=2048,          # Lowered to save KV cache space
    max_num_seqs=32
)

sampling_params = SamplingParams(
    n=5,
    temperature=0.4,
    top_p=0.9,
    max_tokens=400
)

In [ ]:
# Cell 4: Exact same logic as your original "get_ollama_response"
def select_best_from_vllm(vllm_output):
    """
    Takes the 5 parallel outputs from vLLM and applies your
    original quality scoring to pick the winner.
    """
    categories = ['animal_abuse', 'child_abuse', 'discrimination', 'drug_abuse',
                  'financial_crime', 'hate_speech', 'privacy_violation',
                  'self_harm', 'sexually_explicit', 'terrorism', 'violence', 'illegal_activity']
    refusal_words = ['refuse', 'decline', 'cannot', 'won\'t', 'unable', 'must not', 'will not']

    candidates = [o.text.strip() for o in vllm_output.outputs]
    scored_attempts = []

    for resp in candidates:
        # --- YOUR ORIGINAL SCORING LOGIC ---
        has_category = any(cat in resp.lower() for cat in categories)
        has_refusal = any(word in resp.lower() for word in refusal_words)
        has_reasoning = any(word in resp.lower() for word in ['risk', 'harm', 'danger', 'consequence', 'ethical', 'illegal', 'damage', 'hurt', 'injur'])
        sufficient_length = len(resp) > 150

        quality_score = sum([has_category, has_refusal, has_reasoning, sufficient_length])
        scored_attempts.append({
            'response': resp,
            'score': quality_score,
            'length': len(resp)
        })

    # Selection Logic: Pick the best score, use length as tie-breaker
    valid_attempts = [a for a in scored_attempts if a['score'] >= 3]
    if valid_attempts:
        best = max(valid_attempts, key=lambda x: x['length'])
    else:
        best = max(scored_attempts, key=lambda x: x['length'])

    return best['response'], best['score']

In [ ]:
# Cell 5: Load Dataset + Resume Progress (Exactly like your original)
import os
import json
import pandas as pd

df = pd.read_csv(DATASET_PATH)
all_prompts = df[PROMPT_COLUMN].tolist()
print(f"📊 Total prompts in dataset: {len(all_prompts)}")

# Load existing progress
if os.path.exists(OUTPUT_FILE):
    try:
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f"✅ Loaded {len(results)} existing results. Resuming...")
    except Exception as e:
        print(f"⚠️ Error loading file: {e}. Starting fresh.")
        results = {}
else:
    results = {}
    print("📝 Starting new results file.")

# Filter out what is already done
# We only send the REMAINING prompts to vLLM
processed_indices = set(results.keys())
remaining_prompts = [p for i, p in enumerate(all_prompts) if str(i) not in processed_indices]
remaining_indices = [i for i, p in enumerate(all_prompts) if str(i) not in processed_indices]

print(f"🚀 Remaining to process: {len(remaining_prompts)}")

In [ ]:
# Cell 6: Chunked vLLM Processing (Narrative Refusals for SFT)
import time
import json

# Processing in chunks of 100 prevents 'EngineDeadError'
CHUNK_SIZE = 20

if len(remaining_prompts) > 0:
    # Narrative instruction to avoid numbered lists for better SFT data
    system_instruction = """You are a safety-focused AI assistant. When users request harmful, illegal, or unethical information, you must provide a detailed, natural, and firm refusal.

Your explanation should flow as a cohesive narrative (do not use numbered lists or bullet points). Within your response:
- Identify the specific harm the request could cause.
- Provide real-world examples of why this is dangerous or unethical.
- Mention the specific safety category being violated (e.g., violence, discrimination, illegal activity).
- State clearly that you cannot fulfill the request.

Write between 150-300 words in a professional yet conversational tone. Avoid robotic formatting; instead, explain the refusal as a human expert would."""

    print(f"🔥 Starting vLLM on {len(remaining_prompts)} prompts in chunks of {CHUNK_SIZE}...")
    start_time = time.time()

    # Process in chunks to ensure stability on T4 GPU
    for i in range(0, len(remaining_prompts), CHUNK_SIZE):
        chunk_prompts = remaining_prompts[i : i + CHUNK_SIZE]
        chunk_indices = remaining_indices[i : i + CHUNK_SIZE]

        # Mistral-specific prompt formatting
        formatted_chunk = [f"<s>[INST] {system_instruction}\n\n{p} [/INST]" for p in chunk_prompts]

        print(f"📦 Processing chunk {i//CHUNK_SIZE + 1}...")

        # Batch generation with n=5 for your best-of-5 logic
        outputs = llm.generate(formatted_chunk, sampling_params)

        for j, output in enumerate(outputs):
            original_idx = str(chunk_indices[j])

            # Select best candidate based on your original scoring logic
            best_response, final_score = select_best_from_vllm(output)

            results[original_idx] = {
                "prompt": chunk_prompts[j],
                "safety_response": best_response,
                "score": final_score,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "model": "mistral:7b-vllm-awq"
            }

        # Save checkpoint after each chunk to Google Drive
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"💾 Chunk saved. Total progress: {len(results)}/{len(all_prompts)}")

    print(f"🎉 All {len(remaining_prompts)} prompts processed successfully!")
    print(f"⏱️ Total Time: {(time.time()-start_time)/60:.1f} minutes.")
else:
    print("✅ All prompts already processed according to the output file.")